**Cell #01**

# LRM11 -- Stage 2: Ask Sample Questions

Runs a small set of hand-picked Yoga-Sūtra questions end to end through the retrieval + generation
pipeline: embed the question, retrieve the best-matching `lrm_child_chunk_table` rows, and have Claude answer strictly
from those excerpts, with page citations.

This is the plain baseline: no hybrid search, no HyDE, no multi-query, no page expansion -- just
`retrieve_chunks()` + `ask_question()`. Each technique gets its own notebook (`stage2_ask_examples2_rerank.ipynb`
onward) that turns exactly one of them on to show its effect in isolation.

In [1]:
# Cell #02
from reusable_code import init_clients, EMBEDDING_MODEL, GENERATION_MODEL
from reusable_code.env import optional_env

# Client construction (Supabase / Voyage / Anthropic) and env-var loading live in
# ./reusable_code/clients.py -- see reusable_code/README.md for the full guide.
clients = init_clients()

**Cell #03**

## Retrieval

In [2]:
# Cell #04
from reusable_code import (
    with_retry as _retry,
    MIN_CONTEXT_CHUNKS,
    NUM_CONTEXT_CHUNKS,
    embed_query,
    retrieve_chunks,
    page_numbers_for_chunk,
)

# embed_query() / retrieve_chunks() / page_numbers_for_chunk() live in
# ./reusable_code/ask/retrieval.py. They use the `clients` bundle from the cell above by
# default, or accept an explicit `clients=` kwarg (useful in tests).

**Cell #05**

## Generation

In [3]:
# Cell #06
from reusable_code import (
    SYSTEM_PROMPT,
    MAX_ANSWER_TOKENS,
    build_context_block,
    extract_short_answer,
    grounding_words,
    ask_question,
)

# build_context_block() / extract_short_answer() / grounding_words() / ask_question()
# live in ./reusable_code/ask/generation.py.

**Cell #07**

## The 5 questions

A mix on purpose: some have a clean Yes/No answer (to exercise the `Short answer:` formatting), some are
open-ended, and one asks for a cross-sūtra comparison so a good answer has to draw on more than one
retrieved chunk.

In [4]:
# Cell #08
SAMPLE_QUESTIONS = [
    # Yes/No -- a clean textual fact, good for checking the Short answer format.
    "Does Yoga-Sūtra I.2 define yoga as the cessation (nirodha) of the fluctuations of the mind?",
    # Yes/No, but the honest answer needs nuance from the Bhāṣya, not just the sūtra text.
    "Is Īśvara, according to the Yoga-Sūtra, the creator of the world?",
    # Open-ended, likely needs more than one chunk.
    "What does Patañjali's text say happens in the presence of someone firmly established in ahiṃsā?",
    # Open-ended, cross-sūtra comparison -- a good test of whether one chunk is enough context.
    "How do the five vṛtti-s (pramāṇa, viparyaya, vikalpa, nidrā, smṛti) differ from one another?",
    # Yes/No, commonly oversimplified -- tests whether the model over-commits to a flat answer.
    "Is prakrāma (right effort) alone sufficient for attaining samādhi, with no other practice needed?",
]

**Cell #09**

## Run all 5 questions

In [5]:
# Cell #10
results = []
for question in SAMPLE_QUESTIONS:
    print(f"Q: {question}")
    result = ask_question(
        question,
        use_hybrid=False, use_hyde=False, use_multi_query=False, expand_to_parents=False,
    )
    results.append(result)

    print(f"  chunks used: {result['chunks_used']} (source(s): {', '.join(result['source_keys'])})")
    print(f"  source pages: {result['source_pages']}")
    if result["short_answer"]:
        print(f"  Short answer: {result['short_answer']}")
    print()
    print(result["answer"])
    print()
    grounding = ", ".join(result["grounding_words"]) if result.get("grounding_words") else "(none)"
    print(f"  grounding words: {grounding}")
    print("-" * 78)

Q: Does Yoga-Sūtra I.2 define yoga as the cessation (nirodha) of the fluctuations of the mind?
  chunks used: 5 (source(s): yogasutra_m_angot_2021)
  source pages: [303]
  Short answer: No

Short answer: No

The excerpts provided do not contain Yoga-Sūtra I.2 or any discussion of that verse defining yoga as citta-vṛtti-nirodha. The excerpts instead concern Yoga-Sūtra I.20 (and its commentary), which discusses the five means (śraddhā, energy, watchful attention/smṛti, samādhi, and prajñā) leading to "non-cognitive mental enstasis," along with related notes on śraddhā, viveka, and the object of knowledge. There is no material here addressing the specific formulation attributed to Yoga-Sūtra I.2, so the excerpts do not provide enough information to answer the question.

  grounding words: tra, instead, five, means, raddh, energy, watchful, attention, non, cognitive, mental, enstasis
------------------------------------------------------------------------------
Q: Is Īśvara, according to t

**Cell #11**

## Summary table

In [6]:
# Cell #12
print(f"{'#':<3} {'short answer':<13} {'chunks':>7} {'pages':>6} {'sources':<12} question")
for i, r in enumerate(results, start=1):
    short = r["short_answer"] or "n/a"
    print(f"{i:<3} {short:<13} {r['chunks_used']:>7} {len(r['source_pages']):>6} "
          f"{','.join(r['source_keys']):<12} {r['question']}")

#   short answer   chunks  pages sources      question
1   No                  5      1 yogasutra_m_angot_2021 Does Yoga-Sūtra I.2 define yoga as the cessation (nirodha) of the fluctuations of the mind?
2   No                  5      1 yogasutra_m_angot_2021 Is Īśvara, according to the Yoga-Sūtra, the creator of the world?
3   n/a                 5      3 yogasutra_m_angot_2021 What does Patañjali's text say happens in the presence of someone firmly established in ahiṃsā?
4   n/a                 5      1 yogasutra_m_angot_2021 How do the five vṛtti-s (pramāṇa, viparyaya, vikalpa, nidrā, smṛti) differ from one another?
5   No                  5      3 yogasutra_m_angot_2021 Is prakrāma (right effort) alone sufficient for attaining samādhi, with no other practice needed?


**Cell #13**

## Save workspace to GitHub

Synchronize this notebook, answers, and any code changes to GitHub (with auto lock recovery and conflict
resolution).

In [7]:
# Cell #14
from reusable_code import save_to_github

save_to_github("stage2_ask_examples1.ipynb - answers verified and synced")

  RAG11 -> GitHub Robust Sync Utility
  Directory : /Users/mgtimber/CV26/RAG11
  Remote URL: https://github.com/fotomain/RAG11-nutriciology.git
[0/5] Checking repository health & clearing stale locks...
[1/5] Git repository verified.
[2/5] Origin remote verified: https://github.com/fotomain/RAG11-nutriciology.git
[3/5] Staging workspace files...
[4/5] Committing changes: "stage2_ask_examples1.ipynb - answers verified and synced"
[main ee310212] stage2_ask_examples1.ipynb - answers verified and synced
 6 files changed, 127 insertions(+), 47 deletions(-)
 delete mode 100755 py/run/run2b_lrm_chunks.command
[5/5] Synchronizing with GitHub (main)...
      Pushing to origin main (attempt 1/3)...
branch 'main' set up to track 'origin/main'.

  Successfully synchronized with GitHub!
  Branch    : main
  Commit    : ee310212
  Repository: https://github.com/fotomain/RAG11-nutriciology



To https://github.com/fotomain/RAG11-nutriciology.git
   c6dd068b..ee310212  main -> main



True